# Enum — Nível Básico

**Objetivo:** substituir strings e números "mágicos" por constantes nomeadas, sabendo acessar, converter, iterar,
comparar e despachar com `match/case`.

Teoria completa em [`enum.md`](enum.md). Roteiro:

1. O problema das strings mágicas
2. Primeiro Enum: `name` e `value`
3. Convertendo: por valor `Status(...)` e por nome `Status[...]`
4. Iteração, tamanho e pertinência
5. Comparação: `is`, `==` e a diferença para o valor bruto
6. `auto()`
7. Aliases e `@unique`
8. Despacho com `match/case`
9. Valores que não são números
10. Resumo

## 1. O problema das strings mágicas

In [1]:
def ship(order: dict) -> str:
    if order["status"] == "paid":
        return "enviando"
    if order["status"] == "shiped":          # typo: nunca vai casar, e nada avisa
        return "já enviado"
    return "aguardando pagamento"

print(ship({"status": "paid"}))
print(ship({"status": "shipped"}))         # deveria ser "já enviado"
print(ship({"status": "Paid"}))            # maiúscula: cai no default

enviando
aguardando pagamento
aguardando pagamento


Três bugs silenciosos em seis linhas. Nenhum dá erro; todos produzem o resultado errado.

## 2. Primeiro Enum: `name` e `value`

In [2]:
from enum import Enum

class OrderStatus(Enum):
    PENDING = "pending"
    PAID = "paid"
    SHIPPED = "shipped"
    CANCELLED = "cancelled"

s = OrderStatus.PAID
print(s)                 # representação legível
print(repr(s))           # <Classe.NOME: valor>
print(s.name, "|", s.value)
print(type(s), isinstance(s, OrderStatus))

OrderStatus.PAID
<OrderStatus.PAID: 'paid'>
PAID | paid
<enum 'OrderStatus'> True


Com o Enum, o typo vira **erro imediato** (e o editor/type checker avisa antes mesmo de rodar):

In [3]:
try:
    OrderStatus.SHIPED
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: type object 'OrderStatus' has no attribute 'SHIPED'


## 3. Convertendo: por valor e por nome

Dados de fora (JSON, banco, formulário) chegam como valor bruto. Converta **na entrada**:

| sintaxe | busca por | uso |
|---|---|---|
| `OrderStatus("paid")` | **valor** | dados persistidos / JSON / API |
| `OrderStatus["PAID"]` | **nome** | config, argumentos de CLI |

In [4]:
print(OrderStatus("paid"))
print(OrderStatus["SHIPPED"])

try:
    OrderStatus("PAID")                 # valor é case-sensitive
except ValueError as e:
    print("ValueError:", e)
try:
    OrderStatus["paid"]                 # nome também
except KeyError as e:
    print("KeyError:", e)

OrderStatus.PAID
OrderStatus.SHIPPED
ValueError: 'PAID' is not a valid OrderStatus
KeyError: 'paid'


Membros são **singletons**: toda conversão devolve exatamente o mesmo objeto.

In [5]:
print(OrderStatus("paid") is OrderStatus.PAID is OrderStatus["PAID"])

True


## 4. Iteração, tamanho e pertinência

In [6]:
print(list(OrderStatus))
print(len(OrderStatus))
print([s.value for s in OrderStatus])
print({s.name: s.value for s in OrderStatus})

print(OrderStatus.PAID in OrderStatus)
print("paid" in OrderStatus, "  <- por valor, Python 3.12+")
print("refunded" in OrderStatus)

[<OrderStatus.PENDING: 'pending'>, <OrderStatus.PAID: 'paid'>, <OrderStatus.SHIPPED: 'shipped'>, <OrderStatus.CANCELLED: 'cancelled'>]
4
['pending', 'paid', 'shipped', 'cancelled']
{'PENDING': 'pending', 'PAID': 'paid', 'SHIPPED': 'shipped', 'CANCELLED': 'cancelled'}
True
True   <- por valor, Python 3.12+
False


Isso é ótimo para montar **opções de UI**, validar entradas e documentar o que existe:

In [7]:
def parse_status(raw: str) -> OrderStatus:
    if raw not in OrderStatus:
        options = ", ".join(s.value for s in OrderStatus)
        raise ValueError(f"status {raw!r} inválido. Opções: {options}")
    return OrderStatus(raw)

print(parse_status("shipped"))
try:
    parse_status("shiped")
except ValueError as e:
    print(e)

OrderStatus.SHIPPED
status 'shiped' inválido. Opções: pending, paid, shipped, cancelled


## 5. Comparação: `is`, `==` e o valor bruto

* Como membros são singletons, a comparação idiomática é **`is`** (identidade).
* `==` também funciona entre membros.
* Com `Enum` puro, **membro ≠ valor bruto**: `OrderStatus.PAID == "paid"` é `False`. Isso é proposital — força a conversão na borda.
  (No intermediário veremos `StrEnum`, que muda esse comportamento.)

In [8]:
s = OrderStatus("paid")
print("is      :", s is OrderStatus.PAID)
print("==      :", s == OrderStatus.PAID)
print("== 'paid':", s == "paid", "  <- Enum puro não se compara com o valor bruto")
print(".value  :", s.value == "paid")

is      : True
==      : True
== 'paid': False   <- Enum puro não se compara com o valor bruto
.value  : True


Enums são **hashable**: servem como chave de dicionário — o jeito clássico de trocar cadeias de `if/elif` por uma tabela.

In [9]:
LABELS_PT = {
    OrderStatus.PENDING: "Aguardando pagamento",
    OrderStatus.PAID: "Pago",
    OrderStatus.SHIPPED: "Enviado",
    OrderStatus.CANCELLED: "Cancelado",
}
print(LABELS_PT[OrderStatus("shipped")])

Enviado


## 6. `auto()`

Quando o valor não importa (só a identidade do membro), `auto()` gera os valores. Em `Enum` puro gera inteiros 1, 2, 3...

In [10]:
from enum import auto

class Priority(Enum):
    LOW = auto()
    MEDIUM = auto()
    HIGH = auto()
    CRITICAL = auto()

print([(p.name, p.value) for p in Priority])

[('LOW', 1), ('MEDIUM', 2), ('HIGH', 3), ('CRITICAL', 4)]


> ⚠️ Se esses números forem **gravados** em banco/arquivo, inserir um membro no meio muda os valores dos seguintes e corrompe
> os dados antigos. Para valores persistidos, escreva-os explicitamente (ou use `StrEnum` + `auto()`, que gera o nome — intermediário).

## 7. Aliases e `@unique`

Dois nomes com o **mesmo valor** não criam dois membros: o segundo vira **alias** do primeiro e não aparece na iteração.
Às vezes é intencional (nome antigo mantido por compatibilidade), às vezes é um bug de copiar-e-colar.

In [11]:
class Size(Enum):
    SMALL = 1
    MEDIUM = 2
    LARGE = 3
    M = 2            # alias de MEDIUM

print(list(Size))
print(Size.M, Size.M is Size.MEDIUM)
print(list(Size.__members__))     # __members__ inclui aliases

[<Size.SMALL: 1>, <Size.MEDIUM: 2>, <Size.LARGE: 3>]
Size.MEDIUM True
['SMALL', 'MEDIUM', 'LARGE', 'M']


In [12]:
from enum import unique

try:
    @unique
    class Plan(Enum):
        FREE = "free"
        PRO = "pro"
        TEAM = "pro"          # copiar-e-colar
except ValueError as e:
    print("ValueError:", e)

ValueError: duplicate values found in <enum 'Plan'>: TEAM -> PRO


## 8. Despacho com `match/case`

`match` com membros de Enum deixa o fluxo explícito. Sempre inclua `case _:` (ou, no avançado, `assert_never` para o type checker
garantir que todos os casos foram tratados).

In [13]:
def next_action(status: OrderStatus) -> str:
    match status:
        case OrderStatus.PENDING:
            return "cobrar cliente"
        case OrderStatus.PAID:
            return "separar e enviar"
        case OrderStatus.SHIPPED:
            return "acompanhar entrega"
        case OrderStatus.CANCELLED:
            return "estornar se necessário"
        case _:
            raise ValueError(f"status desconhecido: {status!r}")

for s in OrderStatus:
    print(f"{s.name:<10} -> {next_action(s)}")

PENDING    -> cobrar cliente
PAID       -> separar e enviar
SHIPPED    -> acompanhar entrega
CANCELLED  -> estornar se necessário


Vários membros no mesmo `case` com `|`:

In [14]:
def is_final(status: OrderStatus) -> bool:
    match status:
        case OrderStatus.SHIPPED | OrderStatus.CANCELLED:
            return True
        case _:
            return False

print({s.name: is_final(s) for s in OrderStatus})

{'PENDING': False, 'PAID': False, 'SHIPPED': True, 'CANCELLED': True}


## 9. Valores que não são números

O valor pode ser qualquer objeto: string, tupla, até um objeto mais rico. Escolha valores **estáveis e significativos**
quando eles saem do programa (JSON, banco).

In [15]:
class Color(Enum):
    RED = "R"
    GREEN = "G"
    BLUE = "B"

class HttpMethod(Enum):
    GET = ("GET", False)       # (verbo, tem corpo?)
    POST = ("POST", True)
    DELETE = ("DELETE", False)

def create_car(color: Color) -> str:
    match color:
        case Color.RED:
            return "Um carro vermelho"
        case Color.GREEN:
            return "Um carro verde"
        case Color.BLUE:
            return "Um carro azul"
        case _:
            return f"A cor {color!r} não existe aqui"

print(create_car(Color("B")))
print(HttpMethod.POST.value, "| tem corpo?", HttpMethod.POST.value[1])

Um carro azul
('POST', True) | tem corpo? True


> Repare em `case _: return f"A cor {color!r} ..."`: use a **variável** recebida, não a classe (`{Color}` imprimiria `<enum 'Color'>`).
> No intermediário, membros com vários atributos ficam bem mais legíveis com `__init__` (`HttpMethod.POST.has_body`).

## 10. Resumo

| conceito | lembrete |
|---|---|
| membro | singleton com `.name` e `.value` |
| entrada | `Enum(valor)` para dados externos; `Enum[nome]` para config/CLI |
| comparação | `is` (idiomático) ou `==` entre membros; `Enum` puro ≠ valor bruto |
| iteração | `list(E)`, `len(E)`, `valor in E` (3.12+) |
| `auto()` | gera valores; cuidado se forem persistidos |
| aliases | mesmo valor = mesmo membro; `@unique` proíbe |
| tabelas | Enum é hashable → `dict[Enum, ...]` substitui `if/elif` |
| `match/case` | despacho explícito; sempre trate o `_` |

### Exercícios
1. Crie `PaymentMethod` (`pix`, `card`, `boleto`) e uma tabela `dict[PaymentMethod, float]` com a taxa de cada um.
2. Escreva `parse_priority(raw: str)` que aceite o **nome** (`"HIGH"`) e devolva o membro, com erro listando as opções.
3. Crie um Enum com um alias intencional (`COLOUR`/`COLOR`) e verifique o que aparece em `list()` e em `__members__`.

➡️ Próximo: [`02_enum_intermediario.ipynb`](02_enum_intermediario.ipynb)